In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.2 RMSNorm

LayerNorm centers (subtract mean) then scales (divide by std). RMSNorm skips the
centering and just divides by the root-mean-square. Fewer ops, no bias, and in
practice no loss of quality, which is why modern models use it.

In [ ]:
from pocketlm import RMSNorm
from torch import nn

x = torch.randn(2, 4, 8) * 5 + 3
rms = RMSNorm(8)(x)
ln = nn.LayerNorm(8)(x)
print("RMSNorm output RMS ~1:", round(rms.pow(2).mean(-1).sqrt().mean().item(), 4))
print("RMSNorm params:", sum(p.numel() for p in RMSNorm(8).parameters()),
      " LayerNorm params:", sum(p.numel() for p in nn.LayerNorm(8).parameters()))

RMSNorm has only the scale weight (8 params here); LayerNorm also has a bias (16
total). The saving is small per layer but adds up, and the simpler op is faster.

In [ ]:
rms_out = RMSNorm(8)(x)
assert abs(rms_out.pow(2).mean(-1).sqrt().mean().item() - 1.0) < 0.05
assert sum(p.numel() for p in RMSNorm(8).parameters()) < \
       sum(p.numel() for p in nn.LayerNorm(8).parameters())